# Episode 24: Telemetry

Your agents work — but in production you need to *see* what they do: which tools fired, how many tokens you spent, where the time went.

**In this episode you'll build:** observability into a running agent with two complementary mechanisms — in-process events and OpenTelemetry spans.

This is the start of **Group 6 — Production**.

## Two ways to see inside an agent

AG2 Beta gives you two telemetry mechanisms, and they answer different questions:

| | `MemoryStream` events | `TelemetryMiddleware` spans |
|---|---|---|
| Scope | In-process, this Python program | Exported to an external backend |
| Good for | Live UI updates, quick debugging, custom logic | Production tracing — Jaeger, Grafana, Datadog, Honeycomb |
| Format | Python event objects you subscribe to | OpenTelemetry spans (GenAI semantic conventions) |

Use the **stream** when your own code needs to react to what the agent is doing. Use the **middleware** when an ops team needs dashboards and traces.

## Mechanism 1: live events with `MemoryStream`

Every agent run publishes events to a stream — `ToolCallEvent`, `ModelResponse`, and more. Normally the agent makes its own stream internally; pass your own `MemoryStream` to `ask(...)` and you can subscribe to those events *before* the run starts.

`stream.where(EventType).subscribe()` registers a handler. Here we count tool calls and sum token usage across every LLM call.

In [1]:
from dotenv import load_dotenv

load_dotenv()

from autogen.beta import Agent, MemoryStream
from autogen.beta.config import OpenAIConfig
from autogen.beta.events import ModelResponse, ToolCallEvent

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)


def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"{city}: 21C, sunny."


agent = Agent(
    "weather_bot",
    prompt="Use get_weather for weather questions.",
    config=config,
    tools=[get_weather],
)

stream = MemoryStream()
totals = {"llm_calls": 0, "tokens": 0}


@stream.where(ToolCallEvent).subscribe()
def on_tool(event: ToolCallEvent) -> None:
    print(f"[tool call]  {event.name}")


@stream.where(ModelResponse).subscribe()
def on_llm(event: ModelResponse) -> None:
    totals["llm_calls"] += 1
    totals["tokens"] += event.usage.total_tokens


reply = await agent.ask("What's the weather in Paris?", stream=stream)
print(f"[summary]    {totals['llm_calls']} LLM calls, {totals['tokens']} tokens")
print("reply:", reply.body)

[tool call]  get_weather


[summary]    2 LLM calls, 179 tokens
reply: The weather in Paris is currently 21°C and sunny.


## Mechanism 2: OpenTelemetry spans with `TelemetryMiddleware`

For production, you want traces in a real backend. `TelemetryMiddleware` emits **OpenTelemetry spans** following the [GenAI semantic conventions](https://opentelemetry.io/docs/specs/semconv/gen-ai/) — so they export to Jaeger, Grafana Tempo, Datadog, Honeycomb, and others unchanged.

It needs the `tracing` extra:

In [2]:
!pip install "ag2[openai,tracing]" -q

Attach it as a middleware. Below we use an **in-memory exporter** so we can inspect the spans right here; in production you'd swap in an OTLP exporter pointed at your backend.

Each `ask()` produces one `agent` span (`invoke_agent`) with child `llm` spans (`chat`) and `tool` spans (`execute_tool`).

In [3]:
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

from autogen.beta.middleware.builtin import TelemetryMiddleware

exporter = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))

traced_agent = Agent(
    "weather_bot",
    prompt="Use get_weather for weather questions.",
    config=config,
    tools=[get_weather],
    middleware=[
        TelemetryMiddleware(tracer_provider=provider, agent_name="weather_bot")
    ],
)

reply = await traced_agent.ask("What's the weather in Paris?")
print("reply:", reply.body)

spans = exporter.get_finished_spans()
print(f"captured {len(spans)} spans:")
for s in spans:
    print(f"  {s.name}  [{s.attributes.get('ag2.span.type')}]")

reply: The weather in Paris is currently 21°C and sunny.
captured 4 spans:
  chat gpt-4.1-mini-2025-04-14  [llm]
  execute_tool get_weather  [tool]
  chat gpt-4.1-mini-2025-04-14  [llm]
  invoke_agent weather_bot  [agent]


## Exporting to a real backend

The in-memory exporter is for inspection. In production you point a real exporter at your backend. To watch traces live in your terminal, use the console exporter:

```python
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
```

To ship them to a collector (Jaeger, Tempo, Datadog, …), use the OTLP exporter:

```python
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace.export import BatchSpanProcessor

provider.add_span_processor(
    BatchSpanProcessor(OTLPSpanExporter(endpoint="http://localhost:4317"))
)
```

Nothing about the agent changes — only the exporter.

## Additional content

**Span attributes.** Every span carries standard GenAI attributes: `gen_ai.usage.input_tokens`, `gen_ai.response.model`, `gen_ai.tool.name`, and more. Your backend can chart token spend or tool latency without any custom instrumentation.

**Privacy.** By default spans include message content and tool arguments. Pass `capture_content=False` to `TelemetryMiddleware` to omit them in privacy-sensitive environments.

**Stream vs middleware — together.** They aren't exclusive. Run both: the stream drives a live UI, the middleware feeds your observability backend, from the same agent run.

## What's next

Telemetry tells you what *happened*. Episode 25 introduces **observers** — components that watch the event stream and can *intervene*, blocking a run before something bad happens.